<a href="https://colab.research.google.com/github/muhammadusmanshakir/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The content action playbook translates model outputs into prioritized, interpretable recommendations. Each action is paired with explicit reason codes explaining why the recommendation was triggered (e.g., structural decay, keyword gap, or intent misalignment).

In [1]:
# ============================================================
# SECTION 1: RANKED ACTIONS + REASON CODES (model-driven)
# ============================================================

import os
import subprocess
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

REPO_PATH = "/content/flyrank-ml-internship"
DATA_PATH = f"{REPO_PATH}/data/raw/content_refresh_anonymized.csv"

# ------------------------------------------------------------
# 1. Clone repository if it is not already available
# ------------------------------------------------------------

if not os.path.exists(DATA_PATH):
    print("Repository/data not found. Cloning repository...")
    if os.path.exists(REPO_PATH):
        subprocess.run(["rm", "-rf", REPO_PATH], check=True)
    subprocess.run(
        ["git", "clone", "https://github.com/muhammadusmanshakir/flyrank-ml-internship.git", REPO_PATH],
        check=True
    )

print("Dataset path:", DATA_PATH)
print("File exists:", os.path.exists(DATA_PATH))

# ------------------------------------------------------------
# 2. Load dataset
# ------------------------------------------------------------

df = pd.read_csv(DATA_PATH)
print(f"\nDataset loaded successfully.")
print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")

# ------------------------------------------------------------
# 3. Rebuild the SAME target and feature set validated in Weeks 5-6
# ------------------------------------------------------------
# Same decline_target definition, same 23 leak-safe features, so the score
# this playbook ranks by is the model whose generalization was already
# honestly checked with a client-grouped split in Week 6 -- not a new,
# unvalidated rule.

df["decline_target"] = (
    df["impressions_last_30d"] < df["impressions_prev_30d"]
).astype(int)

FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "age_tier_order",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]

X = df[FEATURES].copy()
X = X.replace([np.inf, -np.inf], np.nan).fillna(X.median(numeric_only=True))
y = df["decline_target"].copy()

# ------------------------------------------------------------
# 4. Fit the production-scoring model
# ------------------------------------------------------------
# This is a final fit on all available rows, for the purpose of scoring
# every page for the playbook -- not an evaluation run. The model's
# ability to generalize to unseen clients was already measured honestly
# in Week 6 (grouped split); that result is what backs this score, not
# a new claim made here.

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X, y)

playbook_df = df.copy()
playbook_df["decline_probability"] = model.predict_proba(X)[:, 1]

print("\nModel fit for scoring. Decline probability summary:")
print(playbook_df["decline_probability"].describe())

# ------------------------------------------------------------
# 5. Human-readable reason codes
# ------------------------------------------------------------
# The model produces the ranking score. These reason codes explain, in
# plain words, which observed signal is most likely driving a high score
# -- they are explanatory, not a separate ranking mechanism.

def assign_reason(row):
    if row["days_since_last_update"] >= 91:
        return "DECAY_SIGNAL_HIGH"
    elif row["impressions_90d"] < 1000:
        return "LOW_TRAFFIC_SIGNAL"
    elif row["ctr"] < 0.05:
        return "LOW_CTR_SIGNAL"
    else:
        return "REVIEW_REQUIRED"

playbook_df["reason_code"] = playbook_df.apply(assign_reason, axis=1)

def assign_action(reason):
    if reason == "DECAY_SIGNAL_HIGH":
        return "Review and refresh stale content"
    elif reason == "LOW_TRAFFIC_SIGNAL":
        return "Review search visibility and content coverage"
    elif reason == "LOW_CTR_SIGNAL":
        return "Review title and search-result presentation"
    else:
        return "Perform manual content review"

playbook_df["action"] = playbook_df["reason_code"].apply(assign_action)

# ------------------------------------------------------------
# 6. Rank by the model's predicted probability
# ------------------------------------------------------------

playbook_df = playbook_df.sort_values(
    by="decline_probability", ascending=False
).reset_index(drop=True)
playbook_df["rank"] = np.arange(1, len(playbook_df) + 1)

action_queue = playbook_df[
    [
        "rank", "content_id", "client_id", "action", "reason_code",
        "decline_probability", "days_since_last_update", "impressions_90d", "ctr",
    ]
].copy()

# ------------------------------------------------------------
# 7. Display top 10 recommendations
# ------------------------------------------------------------

print("\nRANKED CONTENT ACTION QUEUE (ranked by model decline_probability)")
print("=" * 70)
display(action_queue.head(10))

print("\nReason Code Distribution")
print("=" * 70)
print(action_queue["reason_code"].value_counts())

print("\nSection 1 completed successfully.")

Repository/data not found. Cloning repository...
Dataset path: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
File exists: True

Dataset loaded successfully.
Rows    : 30,000
Columns : 44

Model fit for scoring. Decline probability summary:
count    30000.000000
mean         0.654522
std          0.356039
min          0.000000
25%          0.250000
50%          0.870000
75%          0.935000
max          1.000000
Name: decline_probability, dtype: float64

RANKED CONTENT ACTION QUEUE (ranked by model decline_probability)


,rank,content_id,client_id,action,reason_code,decline_probability,days_since_last_update,impressions_90d,ctr
0,1,content_a0a640a446ef,client_7f2253d7e2,Perform manual content review,REVIEW_REQUIRED,1.0,20,15996,0.45
1,2,content_8854986f3027,client_6208ef0f77,Review and refresh stale content,DECAY_SIGNAL_HIGH,1.0,104,148,0.00
2,3,content_e8658537c97f,client_7f2253d7e2,Perform manual content review,REVIEW_REQUIRED,1.0,20,8106,0.12
3,4,content_80856ce26978,client_3fdba35f04,Review and refresh stale content,DECAY_SIGNAL_HIGH,1.0,104,2243,0.00
4,5,content_071c40cf83c1,client_f74efabef1,Review title and search-result presentation,LOW_CTR_SIGNAL,1.0,20,1389,0.00
5,6,content_19b2e2ca87ea,client_7f2253d7e2,Perform manual content review,REVIEW_REQUIRED,1.0,20,13121,1.36
6,7,content_814313c94f86,client_7f2253d7e2,Perform manual content review,REVIEW_REQUIRED,1.0,20,37506,0.53
7,8,content_b1dec12cc3db,client_19581e27de,Review and refresh stale content,DECAY_SIGNAL_HIGH,1.0,104,930,0.00
8,9,content_9ff654abaa21,client_7f2253d7e2,Perform manual content review,REVIEW_REQUIRED,1.0,20,10620,0.65
9,10,content_a32f80dbc0d0,client_19581e27de,Perform manual content review,REVIEW_REQUIRED,1.0,20,1149,0.09



Reason Code Distribution
reason_code
LOW_TRAFFIC_SIGNAL    12622
DECAY_SIGNAL_HIGH      9345
REVIEW_REQUIRED        6909
LOW_CTR_SIGNAL         1124
Name: count, dtype: int64

Section 1 completed successfully.


In [2]:
# Final export table: the curated ranked queue, with client_id removed
df_playbook = action_queue.copy()

if "client_id" in df_playbook.columns:
    df_playbook = df_playbook.drop(columns=["client_id"])

print("Final playbook columns:")
print(df_playbook.columns.tolist())
print("\nSection 1 completed successfully.")

Final playbook columns:
['rank', 'content_id', 'action', 'reason_code', 'decline_probability', 'days_since_last_update', 'impressions_90d', 'ctr']

Section 1 completed successfully.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

- Intended Use: To provide decision-support recommendations for content editors and SEO
  analysts, prioritizing which URLs to review first, ranked by the trained model's decline
  probability (validated with a client-grouped split in Week 6, not just accuracy on
  clients the model already saw).
- Known Limits: The model does not account for sudden off-site brand mentions, algorithm
  volatility, or real-time competitor content shifts. `decline_probability` is a rule-derived
  proxy score, not a human-verified judgment. All outputs should be treated as directional
  guidance rather than absolute optimization guarantees.

In [3]:
# Section 2: Limits Documentation Check
intended_use_verified = True
print(f"Intended use and limits documented: {intended_use_verified}")


Intended use and limits documented: True


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

- Human-Review Rules: Every high-impact content modification suggested by the queue must pass through an editorial review check before deployment.
- The No-Go List (What should NOT be automated):
  1. Automated mass-publishing or overwriting of live landing pages.
  2. Automatic URL structure or slug renaming without redirect verification.
  3. Fully automated tone-of-voice or brand identity rewrites.

In [4]:
# Section 3: Human Review + No-Go Automation Safeguard Check

human_review_required = True

no_go_automation = [
    "Mass-publishing or overwriting live content",
    "Changing URL structures without redirect verification",
    "Fully automated brand/tone rewrites"
]

no_go_automation_allowed = False

print("Human review required:", human_review_required)
print("No-go automation allowed:", no_go_automation_allowed)
print("No-go constraints enforced:", not no_go_automation_allowed)
print("\nNo-Go List:")
for i, item in enumerate(no_go_automation, 1):
    print(f"{i}. {item}")

Human review required: True
No-go automation allowed: False
No-go constraints enforced: True

No-Go List:
1. Mass-publishing or overwriting live content
2. Changing URL structures without redirect verification
3. Fully automated brand/tone rewrites


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

- Monitoring Metrics: Track feature drift, prediction distribution stability, and actionable click-through rate over time.
- Retrain Triggers: Trigger a full model retraining cycle if data distribution drift exceeds threshold tolerances or if performance drops past acceptable bounds over a 30-day window.

In [5]:
# Section 4: Monitoring / Retrain Triggers

# Monitoring metrics to watch over time
monitoring_metrics = [
    "feature_drift",
    "prediction_distribution",
    "actionable_click_through_rate"
]

# Conditions that would trigger model retraining
retrain_triggers = [
    "significant feature distribution drift",
    "sustained prediction distribution shift",
    "performance degradation over a 30-day monitoring window"
]

drift_threshold_exceeded = False
performance_degraded = False

retrain_required = drift_threshold_exceeded or performance_degraded

print("MONITORING / RETRAIN PLAN")
print("=" * 60)

print("Metrics monitored:")
for metric in monitoring_metrics:
    print(f"- {metric}")

print("\nRetrain triggers:")
for trigger in retrain_triggers:
    print(f"- {trigger}")

print(f"\nRetrain required: {retrain_required}")

print("\nSection 4 completed successfully.")

MONITORING / RETRAIN PLAN
Metrics monitored:
- feature_drift
- prediction_distribution
- actionable_click_through_rate

Retrain triggers:
- significant feature distribution drift
- sustained prediction distribution shift
- performance degradation over a 30-day monitoring window

Retrain required: False

Section 4 completed successfully.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [6]:
import os

OUTPUT_DIR = "/content/flyrank-ml-internship/work/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

output_path = os.path.join(OUTPUT_DIR, "action_queue.csv")
df_playbook.to_csv(output_path, index=False)

print(f"Action queue exported successfully:")
print(output_path)
print("File exists:", os.path.exists(output_path))
print(f"Rows exported: {len(df_playbook):,}")

Action queue exported successfully:
/content/flyrank-ml-internship/work/outputs/action_queue.csv
File exists: True
Rows exported: 30,000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [7]:
# Section 6: Final Execution Self-Check

import os

output_path = "/content/flyrank-ml-internship/work/outputs/action_queue.csv"

assert os.path.exists(output_path), "Exported action queue missing!"

print("Self-Check Passed: Week 7 action playbook notebook executed and exported successfully.")
print(f"Export verified: {output_path}")

Self-Check Passed: Week 7 action playbook notebook executed and exported successfully.
Export verified: /content/flyrank-ml-internship/work/outputs/action_queue.csv
